## ADC to P.E. Dataflow

### Collect configuration-files
0. Find the panoseti software commit hash `H` from `sw_info.json` in the target observing run directory.
2. Fetch the following calibration files from commit `H` in the panoseti software:
   - Calibration: `control/quabos/*.json`
   - Pulse-height baselines: `/path/to/obs_*/quabo_ph_baselines.json`
   - Quabo uids 

## Converting the `ph256` PFF Data Product from ADC to P.E. 
Apply the following steps to convert pulse-height frame data from raw ADC units into photoelectron units

0. Re-index pixels to convert from the hardware pixel coordinates (bga or qfp) into x-y spatial pixel coordinates. (These transformations are likely handled by the firmware + production software.)
1. Use `pff` to read the 1D array, then use `numpy` to reshape and cast to `shape=(16,16)` and `dtype=np.int16`, respectively.
2. Read baselines from `quabo_baselines.json`. If necessary, apply the re-index operation to these values. Then cast the 1D array into a `shape=(16,16)` `dtype=np.int16` array.
3. Apply the detector-specific linear transform to the ADC-valued pixels to convert them to p.e. units.

For a given quabo `q`, steps 0-3 above are deterministic for every `ph256` PFF frame it produces.

### Formalizing the ADC to P.E. Transformation with Vectorizable Operations
Letting:
- $\sigma_q: I \rightarrow I^\prime$ be the pixel permutation mapping hardware-encoded images to spatial images.
- $B$ = pixel-level baselines, after any $\sigma_q$ transformations and rotations. 
- $N$ = block matrix array of `n` coefficients for each detector region from `quabo_calib_X.json`
- $M$ = block matrix array of `m` coefficients for each detector region from `quabo_calib_X.json`

The ADC to p.e. transformation for a `ph256` PFF image $I$ is given by:

$$
\begin{align} 
f(I) &= (\sigma_q(I) + B) \odot N + M \\
     &= (\sigma_q(I)\odot N) + (B \odot N + M)
\end{align}$$

where $\odot$ denotes element-wise multiplication.

The main challenge in this transformation is constructing $\sigma_q$, `B`, `N`, and `M` from the various configuration files, both local to the run and externally version-controlled in the panoseti software repo.


## Dataframe Schemas

### quabo_install_df schema
Each record represents a quabo used in a specific observing run.

Columns:
- `dome` = dome name from `obs_config.json` file.
- `module_ip_addr` = `ip` address of the module, unique for a given observing run.
- `mobo_serial_no` = module board serial number on which this quabo was installed.
- *`quabo_uid`* = UID of the quabo hardware.
- `quabo_num` = spatial position of the quabo in its module.
- `detector_overvoltage` = overvoltage setting used in observing run.

Primary keys: (`quabo_uid`)

Dependencies:
- `obs_config.json`
- `quabo_uids.json`


### `quabo_info_df` schema
Each record represents a unique quabo board known to the panoseti software.

Columns:
- *`quabo_uid`* = UID of the quabo hardware.
- `board_version` = board version, one of two values `{"qfp", "bga"}`.
- `serialno_str` = string used in `quabo_info.json` file. e.g. "SN019".
- `serialno` = parsed serial number from `serialno_str`.
- `detector_serialno_i` = serial number of the ith detector array, for `i = 0, 1, 2, 3`.

Primary keys: (`quabo_uid`)

Dependencies:
- `control/quabos/quabo_info.json`

### detector_calibration_df schema
Each record represents a detector-level calibration 


Dependencies:
- `quabo_ph_baseline.json`

In [1]:
from IPython import display
from pathlib import Path
import os
import numpy
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass
from rich import print
from rich.pretty import pprint
import sys
import json

# Import utils from other panoseti directories
repo_root = Path('..')
util_path = repo_root / 'util'
control_utils = repo_root / 'control/utils'
for p in [util_path, control_utils]:
    sys.path.append(str(p))

import pff, config_file, pixel_coords, util

In [2]:
quabo_config_root = repo_root / 'control/quabos'
os.listdir(quabo_config_root)

['detovervol_3v',
 'quabo_info.json',
 'detovervol_2v',
 'quabo_pixelmap_maroc2phys_bga.json',
 'quabo_pixelmap_maroc2phys_qfp.json',
 'detector_info.json',
 '.ipynb_checkpoints',
 'quabo_pixelmap_phys2maroc_qfp.json',
 'quabo_pixelmap_phys2maroc_bga.json']

In [3]:
obs_dir = Path('obs_Lick.start_2024-07-25T04:34:06Z.runtype_sci-data.pffd')
os.listdir(obs_dir)

['start_2024-07-25T04_34_46Z.dp_ph256.bpp_2.module_3.seqno_0.debug_TRUNCATED.pff',
 'recording_ended',
 'daq_config.json',
 'quabo_ph_baseline.json',
 'start_2024-07-25T04_34_46Z.dp_img16.bpp_2.module_1.seqno_0.debug_TRUNCATED.pff',
 'obs_config.json',
 'start_2024-07-25T04_34_46Z.dp_ph256.bpp_2.module_1.seqno_0.debug_TRUNCATED.pff',
 '.ipynb_checkpoints',
 'data_config.json',
 'quabo_uids.json']

In [4]:
quabo_uid_path = obs_dir/'quabo_uids.json'
# quabo_uids_df = 
# quabo_uids_df

In [5]:
with open(quabo_uid_path, 'rb') as f:
    quid_j = json.load(f)
    for d_idx, dome in enumerate(quid_j['domes']):
        for m_idx, module in enumerate(dome['modules']):
            (pd.json_normalize(module))
    # quid_j  = pd.read_json(f)
    # quid_df = pd.json_normalize(
    #     quid_df['domes'],
    #     max_level=10
    # )
pprint(quid_j)

{
│   'domes': [
│   │   {
│   │   │   'modules': [
│   │   │   │   {
│   │   │   │   │   'ip_addr': '192.168.0.4',
│   │   │   │   │   'quabos': [
│   │   │   │   │   │   {'uid': '6ae310480d05824'},
│   │   │   │   │   │   {'uid': '6ae310480d08c24'},
│   │   │   │   │   │   {'uid': '6ae310480d03022'},
│   │   │   │   │   │   {'uid': '6ae310480d07024'}
│   │   │   │   │   ]
│   │   │   │   }
│   │   │   ]
│   │   },
│   │   {
│   │   │   'modules': [
│   │   │   │   {
│   │   │   │   │   'ip_addr': '192.168.0.12',
│   │   │   │   │   'quabos': [
│   │   │   │   │   │   {'uid': '6ae310480d0a425'},
│   │   │   │   │   │   {'uid': '6a238a180e0b417'},
│   │   │   │   │   │   {'uid': '6ae310480d0c024'},
│   │   │   │   │   │   {'uid': '6a238a180e0b416'}
│   │   │   │   │   ]
│   │   │   │   }
│   │   │   ]
│   │   }
│   ]
}